# Fase 01 : Limpieza y primera exploración de datos 
**Institución:** Universidad Católica Boliviana "San Pablo"  
**Programa:** Maestría en Ciencia de Datos  
**Autor:** Carlos Andrés Pérez Guzmán  
**Fecha:** Septiembre 2026  

---
**Entorno y Objetivos del Notebook**
> **Nota:** Limpieza, y selección de casos de interés 

* **Fase del Proyecto:** Primera fase
* **Core Tech Stack:** `Python` (Programación general para ciencia de datos)
---

<h3 style="color:#05856d">1. Librerías</h3>
<hr>

In [ ]:
from pathlib import Path

import polars as pl
import polars.selectors as cs
from pyprojroot import here


<h3 style="color:#05856d">2. Carga de información</h3>
<hr>

Se importan las funciones de carga y limpieza de datos

In [2]:
from s_01_functions import limpiar_texto_own, load_data_own

df = (
    load_data_own( file_path_param = 'iowa_liquor_sales_2026_1263_rows.csv')
    .with_columns( limpiar_texto_own(column_param = cs.string() - cs.by_name('ordered_on')),
                   pl.col('ordered_on').str.to_date(format = '%Y-%m-%d') )
    .with_columns( pl.col('im_desc','category_name').cast(dtype = pl.Categorical()),
                   pl.col('sales_bottles').cast(dtype = pl.Int64()) ) )

df.glimpse()

Iniciando carga de datos locales desde: iowa_liquor_sales_2026_1263_rows.csv
Carga completada: 964947 registros en 0.43 segundos.
Rows: 964947
Columns: 21
$ bottle_volume_ml  <f64> 750.0, 1750.0, 375.0, 50.0, 1750.0, 375.0, 375.0, 750.0, 750.0, 50.0
$ category_code     <i64> 1011100, 1012100, 1012100, 1012100, 1062400, 1062400, 1011200, 1011200, 1011200, 1011200
$ category_name     <cat> BLENDED WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, SPICED RUM, SPICED RUM, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES
$ county_fips_code  <i64> 19177, 19177, 19177, 19177, 19177, 19177, 19177, 19177, 19177, 19177
$ county_name       <str> 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN'
$ im_desc           <cat> RED STAG BLACK CHERRY, BLACK VELVET, BLACK VELVET, CROWN ROYAL MINI, ADMIRAL NELSON SPICED, ADMIRAL NELSON SPICED, JIM BEAM, JI

<h3 style="color:#05856d">3. Selección de variables y transformaciones particulares</h3>
<hr>

Es importante identificar qué variables tienen alguna funcionalidad estratégica al construir un modelo de decisión.

Se elimina la categoría del producto debido a que no es consistente con el número de producto (item_no)

In [3]:
# Variables con utilidad estratégica
df_selected_variables = [ 'ordered_on','item_no','im_desc','sales_bottles','store_no',
                          'category_name','sales_dollars','pack','bottle_volume_ml' ]

# Variables sin utilidad estratégica
df_eliminated_variables = set(df.columns) - set(df_selected_variables)
print('Variables sin utilidad estratégica:', *sorted(df_eliminated_variables), '\n', sep = '\n')

# Selección de variables de interés
df_01 = df.select(df_selected_variables)
df_01.glimpse()

Variables sin utilidad estratégica:
category_code
county_fips_code
county_name
invoice_id
sales_gallons
sales_liters
store_address
store_city
store_name
store_zip_code
vendor_name
vendor_number


Rows: 964947
Columns: 9
$ ordered_on       <date> 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18
$ item_no           <i64> 27544, 11788, 11774, 11290, 43028, 43024, 19064, 27783, 27410, 19061
$ im_desc           <cat> RED STAG BLACK CHERRY, BLACK VELVET, BLACK VELVET, CROWN ROYAL MINI, ADMIRAL NELSON SPICED, ADMIRAL NELSON SPICED, JIM BEAM, JIM BEAM APPLE, JIM BEAM HONEY, JIM BEAM MINI
$ sales_bottles     <i64> 2, 6, 7, 2, 12, 8, 7, 2, 2, 2
$ store_no          <i64> 4296, 4296, 4296, 4296, 4296, 4296, 4296, 4296, 4296, 4296
$ category_name     <cat> BLENDED WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, SPICED RUM, SPICED RUM, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKI

Revisar integridad de la llave del producto

In [4]:
duplicados_im_desc = (
    df_01
    .select('item_no', 'im_desc')
    .unique()
    .filter(pl.col('im_desc').is_duplicated())
    .sort('im_desc','item_no')
    )

print(f"Duplicados por la descripción del producto (im_desc): {duplicados_im_desc.height > 0}. Casos: {duplicados_im_desc.height}")
print(duplicados_im_desc.head())

duplicados_item_no = (
    df_01
    .select('item_no', 'im_desc')
    .unique()
    .filter(pl.col('item_no').is_duplicated())
    .sort('item_no','im_desc')
    )

print(f"Duplicados por número de ítem (item_no): {duplicados_item_no.height > 0}. Casos: {duplicados_item_no.height}")
print(duplicados_item_no.head())


duplicados_compuesto = (
    df_01
    .select('item_no','im_desc')
    .unique()
    .is_duplicated().any()
    )

print(f"\nDuplicados por la descripción del producto (im_desc) y número de item (item_no): {duplicados_compuesto}")


Duplicados por la descripción del producto (im_desc): True. Casos: 1072
shape: (5, 2)
┌─────────┬──────────────────────────┐
│ item_no ┆ im_desc                  │
│ ---     ┆ ---                      │
│ i64     ┆ cat                      │
╞═════════╪══════════════════════════╡
│ 21235   ┆ 1792 SMALL BATCH BOURBON │
│ 21239   ┆ 1792 SMALL BATCH BOURBON │
│ 64933   ┆ 1800 COCONUT             │
│ 964931  ┆ 1800 COCONUT             │
│ 89206   ┆ 1800 REPOSADO            │
└─────────┴──────────────────────────┘
Duplicados por número de ítem (item_no): True. Casos: 58
shape: (5, 2)
┌─────────┬─────────────────────────────────┐
│ item_no ┆ im_desc                         │
│ ---     ┆ ---                             │
│ i64     ┆ cat                             │
╞═════════╪═════════════════════════════════╡
│ 1799    ┆ CAPTAIN MORGAN ORIGINAL SPICED… │
│ 1799    ┆ CAPTAIN MORGAN ORIGINAL SPICED… │
│ 7224    ┆ MACALLAN SHERRY OAK 12YO 110 P… │
│ 7224    ┆ MACALLAN SHERRY OAK 12YO 110PF  │


Aplicar una llave consistente

In [5]:
df_01 = (
    df_01
    .with_columns(
        pl.format(
            'PRD_{}',
            pl.struct(['item_no', 'im_desc'])
            .rank(method = 'dense')
            .cast(pl.String)
            .str.zfill(6) ).alias('product_code'))
    .pipe(lambda df: df.select(sorted(df.columns)))
    )

df_01.glimpse()

Rows: 964947
Columns: 10
$ bottle_volume_ml  <f64> 750.0, 1750.0, 375.0, 50.0, 1750.0, 375.0, 375.0, 750.0, 750.0, 50.0
$ category_name     <cat> BLENDED WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, SPICED RUM, SPICED RUM, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES
$ im_desc           <cat> RED STAG BLACK CHERRY, BLACK VELVET, BLACK VELVET, CROWN ROYAL MINI, ADMIRAL NELSON SPICED, ADMIRAL NELSON SPICED, JIM BEAM, JIM BEAM APPLE, JIM BEAM HONEY, JIM BEAM MINI
$ item_no           <i64> 27544, 11788, 11774, 11290, 43028, 43024, 19064, 27783, 27410, 19061
$ ordered_on       <date> 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18
$ pack              <i64> 12, 6, 24, 10, 6, 24, 24, 12, 12, 12
$ product_code      <str> 'PRD_000933', 'PRD_000201', 'PRD_000197', 'PRD_000177', 'PRD_001529', 'PRD_001525', 'PRD_000506', 'PRD_000951', 'PRD_00092

Revisar datos faltantes

In [6]:
null_variables = (
    df_01
    .select( 
        pl.all().null_count().name.suffix('_nulos'),
        (pl.all().null_count() / pl.len() * 100).name.suffix('_pct'))
    .unpivot()
    .with_columns(
        columna = pl.col('variable').str.replace('_(nulos|pct)$', ''),
        metrica = pl.col("variable").str.extract('_(nulos|pct)$', 1) )
    .pivot(index = 'columna', on = 'metrica', values = 'value')
    .sort('pct', descending = True)
    )

print(null_variables)

shape: (10, 3)
┌──────────────────┬───────┬─────┐
│ columna          ┆ nulos ┆ pct │
│ ---              ┆ ---   ┆ --- │
│ str              ┆ f64   ┆ f64 │
╞══════════════════╪═══════╪═════╡
│ bottle_volume_ml ┆ 0.0   ┆ 0.0 │
│ category_name    ┆ 0.0   ┆ 0.0 │
│ im_desc          ┆ 0.0   ┆ 0.0 │
│ item_no          ┆ 0.0   ┆ 0.0 │
│ ordered_on       ┆ 0.0   ┆ 0.0 │
│ pack             ┆ 0.0   ┆ 0.0 │
│ product_code     ┆ 0.0   ┆ 0.0 │
│ sales_bottles    ┆ 0.0   ┆ 0.0 │
│ sales_dollars    ┆ 0.0   ┆ 0.0 │
│ store_no         ┆ 0.0   ┆ 0.0 │
└──────────────────┴───────┴─────┘


<h3 style="color:#05856d">4. Exportar datos limpios</h3>
<hr>

In [7]:
df_01.write_parquet( file = Path(here(),'01_cleaned_data','df_01.parquet'), 
                     compression = 'zstd',
                     compression_level = 12 )